In [2]:

# =======================
# ② 依赖导入
# =======================
import re
from openpyxl import load_workbook

# =======================
# ① 文件路径配置（放最前）
# =======================
INPUT_FILE = r"地址去除/原文件/高途普通批_2023_宁夏.xlsx"
OUTPUT_FILE = r"地址去除/输出结果/高途普通批_2023_宁夏_地址处理后.xlsx"
SHEET_NAME = None  # None = 默认第一个 sheet

TARGET_HEADER = "院校名专业组代码/专业组代码"
NEW_HEADER = "院校名专业组代码/专业组代码_清洗"


# =======================
# ③ 正则与规则定义
# =======================

# 中英文括号
PAREN_PATTERN = re.compile(r'(\([^()]*\)|（[^（）]*）)')

# 地址关键词
ADDR_KEYWORDS = [
    "省", "市", "区", "县", "旗", "镇", "乡", "街道", "街",
    "路", "道", "号", "村", "社区", "新区",
    "开发区", "高新区", "工业园", "科技园", "园区"
]

# 典型行政区结构
ADDR_REGEX = re.compile(
    r"("
    r"[\u4e00-\u9fff]{2,}(省|自治区|特别行政区|市)"
    r"[\u4e00-\u9fff]{0,}(市|州|盟|地区)?"
    r"[\u4e00-\u9fff]{1,}(区|县|旗)"
    r")"
)


# =======================
# ④ 核心函数
# =======================

def is_address_text(paren_segment: str) -> bool:
    """判断括号内容是否为地址信息"""
    if paren_segment is None:
        return False

    s = str(paren_segment).strip("()（）").strip()
    s = re.sub(r"^\s*注\s*[:：]\s*", "", s)

    if ADDR_REGEX.search(s):
        return True

    if any(k in s for k in ADDR_KEYWORDS):
        return True

    return False


def remove_address_parentheses(cell_value) -> str:
    """删除括号中是地址的内容（连括号一起删）"""
    if cell_value is None:
        return ""

    text = str(cell_value)

    def repl(m):
        seg = m.group(0)
        return "" if is_address_text(seg) else seg

    cleaned = PAREN_PATTERN.sub(repl, text)
    cleaned = re.sub(r"\s+", " ", cleaned).strip()
    return cleaned


def find_header_col(ws, header_name: str) -> int:
    """查找表头列号"""
    for col in range(1, ws.max_column + 1):
        val = ws.cell(row=1, column=col).value
        if val and str(val).strip() == header_name:
            return col
    raise ValueError(f"未找到表头列：{header_name}")


# =======================
# ⑤ 主处理逻辑
# =======================

def main():
    wb = load_workbook(INPUT_FILE)
    ws = wb[SHEET_NAME] if SHEET_NAME else wb.active

    target_col = find_header_col(ws, TARGET_HEADER)
    insert_col = target_col + 1

    # 在目标列后插入新列
    ws.insert_cols(insert_col)
    ws.cell(row=1, column=insert_col).value = NEW_HEADER

    for r in range(2, ws.max_row + 1):
        raw = ws.cell(row=r, column=target_col).value
        ws.cell(row=r, column=insert_col).value = remove_address_parentheses(raw)

    wb.save(OUTPUT_FILE)
    print("处理完成，输出文件：", OUTPUT_FILE)


# =======================
# ⑥ 程序入口
# =======================
if __name__ == "__main__":
    main()


处理完成，输出文件： 地址去除/输出结果/高途普通批_2023_宁夏_地址处理后.xlsx


In [5]:


# =======================
# ② 依赖导入
# =======================
import re
from openpyxl import load_workbook


# =======================
# ① 文件路径配置（放最前）
# =======================
INPUT_FILE = r"地址去除/原文件/高途普通批_2023_宁夏.xlsx"
OUTPUT_FILE = r"地址去除/输出结果/高途普通批_2023_宁夏_地址处理后.xlsx"
SHEET_NAME = None  # None = 默认第一个 sheet

TARGET_HEADER = "院校名专业组代码/专业组代码"
NEW_HEADER = "院校名专业组代码/专业组代码_清洗"


# =======================
# ③ 地址规则（只删“开头连续地址片段”）
# =======================

# 行政区后缀（从严：只认这些作为“地址片段”的结尾）
ADMIN_SUFFIX = [
    "特别行政区", "自治区",
    "省",
    "市", "州", "盟", "地区",
    "自治州", "自治县",  # 少量省内常见
    "区", "县", "旗",
    "新区", "开发区", "高新区", "工业园", "园区", "科技园"
]

# 把后缀做成一个正则 alternation，按长度降序避免“新区”被“区”抢先匹配
ADMIN_SUFFIX_PATTERN = "|".join(sorted(map(re.escape, ADMIN_SUFFIX), key=len, reverse=True))

# 一个“地址片段”= 若干中文 + 以上后缀
LEADING_ADMIN_RE = re.compile(rf"^\s*([\u4e00-\u9fff]{{2,10}}(?:{ADMIN_SUFFIX_PATTERN}))")

NOTE_PREFIX_RE = re.compile(r"^\s*注\s*[:：]\s*")


def strip_leading_address(text: str) -> str:
    """
    仅删除 text 开头连续出现的地址片段：
    例：
      '天津市色盲、色弱考生不予录取' -> '色盲、色弱考生不予录取'
      '贵州省贵阳市贵安新区患有轻度色觉异常(俗称色弱)' -> '患有轻度色觉异常(俗称色弱)'
      '辽宁省沈阳市于洪区外语只用英语授课不招收色盲色弱考生' -> '外语只用英语授课不招收色盲色弱考生'
    """
    s = text.strip()
    s = NOTE_PREFIX_RE.sub("", s).strip()

    # 反复删“开头地址片段”
    while True:
        m = LEADING_ADMIN_RE.match(s)
        if not m:
            break
        s = s[m.end():].strip()

    # 去掉地址删完后残留的标点开头
    s = s.lstrip("，,、;；:：-— ")
    return s


# =======================
# ④ 括号解析（支持嵌套）
# =======================

def parse_segments_keep_parens(s: str):
    """
    把字符串按“顶层括号段”切分，支持嵌套括号：
    返回 [(is_paren, segment_str), ...]
    is_paren=True 表示 segment_str 是一个完整的顶层括号段（含括号本身）
    """
    res = []
    i = 0
    n = len(s)

    pairs = {"(": ")", "（": "）"}
    open_set = set(pairs.keys())
    close_set = set(pairs.values())

    while i < n:
        ch = s[i]
        if ch in open_set:
            # 收集括号外文本（如有）
            # （在进入这里前，括号外文本应已被收集；这里直接解析括号段）
            opener = ch
            closer = pairs[opener]
            depth = 0
            j = i
            while j < n:
                cj = s[j]
                if cj == opener:
                    depth += 1
                elif cj == closer:
                    depth -= 1
                    if depth == 0:
                        # i..j 是一个完整顶层括号段
                        res.append((True, s[i:j+1]))
                        i = j + 1
                        break
                j += 1
            else:
                # 括号不闭合：当普通文本处理
                res.append((False, s[i:]))
                break
        else:
            # 收集直到下一个括号开始
            j = i
            while j < n and s[j] not in open_set:
                j += 1
            res.append((False, s[i:j]))
            i = j

    return res


def clean_paren_segment(seg: str) -> str:
    """
    清洗一个顶层括号段：
    - 若括号内开头是地址（含/不含注：）-> 删除开头地址片段
    - 删完为空 -> 返回空字符串（整段括号删除）
    - 删完非空 -> 返回 (剩余内容) 保留原括号类型
    - 若括号内开头不是地址 -> 原样返回
    """
    seg = seg.strip()
    if not seg:
        return seg

    left = seg[0]
    right = seg[-1]
    inner = seg[1:-1]

    inner2 = strip_leading_address(inner)

    # 如果发生了变化（说明识别到了地址并剥离）
    if inner2 != inner.strip():
        inner2 = inner2.strip()
        if not inner2:
            return ""  # 纯地址：整段删除
        return f"{left}{inner2}{right}"  # 地址+说明：保留说明
    else:
        return seg  # 非地址开头：原样保留


def clean_cell(text) -> str:
    if text is None:
        return ""

    s = str(text)

    parts = parse_segments_keep_parens(s)
    out = []
    for is_paren, seg in parts:
        if not is_paren:
            out.append(seg)
        else:
            out.append(clean_paren_segment(seg))

    cleaned = "".join(out)
    cleaned = re.sub(r"\s+", " ", cleaned).strip()
    return cleaned


# =======================
# ⑤ Excel 处理
# =======================

def find_header_col(ws, header_name: str) -> int:
    for col in range(1, ws.max_column + 1):
        val = ws.cell(row=1, column=col).value
        if val and str(val).strip() == header_name:
            return col
    raise ValueError(f"未找到表头列：{header_name}")


def main():
    wb = load_workbook(INPUT_FILE)
    ws = wb[SHEET_NAME] if SHEET_NAME else wb.active

    target_col = find_header_col(ws, TARGET_HEADER)
    insert_col = target_col + 1

    ws.insert_cols(insert_col)
    ws.cell(row=1, column=insert_col).value = NEW_HEADER

    for r in range(2, ws.max_row + 1):
        raw = ws.cell(row=r, column=target_col).value
        ws.cell(row=r, column=insert_col).value = clean_cell(raw)

    wb.save(OUTPUT_FILE)
    print("处理完成，输出文件：", OUTPUT_FILE)


if __name__ == "__main__":
    main()


处理完成，输出文件： 地址去除/输出结果/高途普通批_2023_宁夏_地址处理后.xlsx


In [14]:
# =======================
# ② 依赖导入
# =======================
import re
from openpyxl import load_workbook


# =======================
# ① 文件路径配置（放最前）
# =======================
INPUT_FILE = r"地址去除/原文件/高途普通批_2024_云南2.xlsx"
OUTPUT_FILE = r"地址去除/输出结果/2024-云南.xlsx"
SHEET_NAME = None  # None = 默认第一个 sheet

SCHOOL_SRC_HEADER = "schoolUnitTitle"   # 原始院校字段
SCHOOL_CLEAN_HEADER = "schoolUnitTitle_清洗"  # 输出：院校字段清洗结果（去院校名后的括号说明）
MAJOR_SRC_HEADER = "majorSubtitle"                  # 原始专业字段

MAJOR_NAME_HEADER = "专业名称"
MAJOR_REMARK_HEADER = "专业备注"
MAJOR_REMARK_MERGED_HEADER = "专业备注_合并去重"


# =======================
# ③ 院校字段清洗（增强版）
# =======================

# 常见行政后缀（用于“像地址”的判断）
ADMIN_SUFFIX = [
    "特别行政区", "自治区",
    "省",
    "自治州", "自治县",
    "市", "州", "盟", "地区",
    "新区", "开发区", "高新区", "工业园", "园区", "科技园",
    "区", "县", "旗", "镇", "乡", "村", "街道"
]
ADMIN_SUFFIX_PATTERN = "|".join(sorted(map(re.escape, ADMIN_SUFFIX), key=len, reverse=True))
NOTE_PREFIX_RE = re.compile(r"^\s*注\s*[:：]\s*")

# 说明词：出现这些，优先认为“应该保留”（即使里面夹着地址，也只删地址不删说明）
EXPLAIN_WORD_RE = re.compile(
    r"(只招|招收|录取|毕业|报考|不得|不予|不招|禁止|不允许|须|需要|"
    r"建议|外语|语种|英语|授课|课程|培养|安排|交换|升读|转专业|体检|"
    r"色盲|色弱|有效志愿|无效志愿|在校期间|资助|生活补助|就业|服务|年限)"
)

# “像校区/地点/机构名但不是说明”的关键词（剩下只有这些就丢弃）
CAMPUS_ONLY_RE = re.compile(
    r"^(?:"
    r"(?:本校|本部|校本部|主校区|分校区|分校|校区|校园|"
    r"[A-Z]区|[A-Z]校区|"
    r"南校区|北校区|东校区|西校区|"
    r"南校园|北校园|东校园|西校园|"
    r"校区南校园|校区东校园|校区北校园|校区西校园|"
    r"基地|园区|创新港|大学城|科技城|"
    r"深圳|东莞|珠海|海南|雄安|苏州|无锡|江阴|昆山|"
    r"宏福校区|翔安校区|思明校区|仙林校区|鼓楼校区|卫津路校区|北洋园校区"
    r")"
    r"[\s，,;；:：+\-—（）()]*"
    r")+$"
)

# 纯门牌号 / 路号：整段删
PURE_HOUSE_NO_RE = re.compile(r"^\s*[\u4e00-\u9fffA-Za-z0-9\-—＋+、，, ]{0,10}?(?:\d{1,5}号|\d{1,5}号院|\d{1,5}号楼)\s*$")
# 地址强特征：含“省市区县街道路号园区邮编”等
ADDR_STRONG_RE = re.compile(
    rf"({ADMIN_SUFFIX_PATTERN}|路|街|大道|巷|弄|号|号院|号楼|邮编|校址|地址|新城|大学城|"
    r"科技园|工业园|高新|开发区|教育园区|创新试验区|创新港)"
)

# 地址片段（用于从说明里“抠掉地址”但保留说明）
# 目标：删除类似“湖北省武汉市武昌区八一路299号”“广东省广州市…路135号”等
ADDR_SPAN_RE = re.compile(
    rf"(?:(?:[\u4e00-\u9fff]{{2,10}}(?:{ADMIN_SUFFIX_PATTERN}))+\s*)?"
    r"(?:[\u4e00-\u9fff0-9]{1,40})?"
    r"(?:路|街|大道|巷|弄|号|号院|号楼|园区|新区|开发区|科技园|工业园|大学城)"
    r"[\u4e00-\u9fff0-9\-—＋+]{0,30}"
)

# 删除“校区：地址”这种前导地址子句（允许多个，直到遇到明显说明句）
LEADING_ADDR_CLAUSE_RE = re.compile(
    rf"^\s*"
    r"(?:(?:[\u4e00-\u9fff0-9A-Za-z]+?(?:校区|校园|基地|园区|办学点|试验区))\s*[:：])?"
    rf"(?:[\u4e00-\u9fff]{{2,10}}(?:{ADMIN_SUFFIX_PATTERN}))?"
    r"[\u4e00-\u9fff0-9A-Za-z，,;；:：+\-— ]{0,80}?"
    r"(?:路|街|大道|巷|弄|号|园区|新区|开发区|科技园|工业园|大学城)"
    r"[\u4e00-\u9fff0-9A-Za-z\-—＋+ ]{0,30}"
    r"\s*[。；;，,]\s*"
)

def _normalize_space(s: str) -> str:
    s = re.sub(r"\s+", " ", s or "").strip()
    # 清掉括号里常见残渣符号
    s = re.sub(r"[;；:：]\s*(?=[)）])", "", s)
    s = re.sub(r"[+＋]\s*(?=[)）])", "", s)
    s = re.sub(r"[，,]\s*(?=[)）])", "", s)
    s = re.sub(r"[。\.]\s*(?=[)）])", "", s)
    return s.strip()

def parse_segments_keep_parens(s: str):
    """
    顶层括号切分，支持嵌套括号。
    返回 [(is_paren, segment_str), ...]
    """
    res = []
    i = 0
    n = len(s)

    pairs = {"(": ")", "（": "）"}
    open_set = set(pairs.keys())

    while i < n:
        ch = s[i]
        if ch in open_set:
            opener = ch
            closer = pairs[opener]
            depth = 0
            j = i
            while j < n:
                cj = s[j]
                if cj == opener:
                    depth += 1
                elif cj == closer:
                    depth -= 1
                    if depth == 0:
                        res.append((True, s[i:j + 1]))
                        i = j + 1
                        break
                j += 1
            else:
                res.append((False, s[i:]))
                break
        else:
            j = i
            while j < n and s[j] not in open_set:
                j += 1
            res.append((False, s[i:j]))
            i = j

    return res

def dedupe_parens_globally(s: str) -> str:
    parts = parse_segments_keep_parens(s)
    seen = set()
    result = []

    for is_paren, seg in parts:
        if not is_paren:
            result.append(seg)
            continue

        inner = seg[1:-1].strip()
        inner_key = re.sub(r"\s+", "", inner)

        if inner_key in seen:
            continue

        seen.add(inner_key)
        result.append(seg)

    return "".join(result)

def remove_school_prefix_keep_from_first_paren(s: str) -> str:
    """
    去除院校名称：删除第一个 '(' 或 '（' 之前的内容，仅保留从第一个括号开始的内容。
    若无括号：返回空字符串（表示只有院校名，去掉后为空）。
    """
    if not s:
        return ""

    idx1 = s.find("(")
    idx2 = s.find("（")
    idxs = [i for i in [idx1, idx2] if i != -1]
    if not idxs:
        return ""

    cut = min(idxs)
    return s[cut:].strip()

def _strip_address_inside_text(inner: str) -> str:
    """
    从“说明文本”里抠掉地址，但尽量不动说明句。
    """
    s = (inner or "").strip()
    if not s:
        return ""

    s = NOTE_PREFIX_RE.sub("", s).strip()

    # 先反复删掉“前导地址子句”（校区：地址。/ 省市区路号。）
    for _ in range(10):
        ns = LEADING_ADDR_CLAUSE_RE.sub("", s)
        if ns == s:
            break
        s = ns.strip()

    # 再把文本中出现的“地址片段”删掉（不强行删整句）
    s = ADDR_SPAN_RE.sub("", s)

    # 清理重复标点与残渣
    s = re.sub(r"[，,]{2,}", "，", s)
    s = re.sub(r"[；;]{2,}", "；", s)
    s = re.sub(r"[。\.]{2,}", "。", s)

    # 去掉“校区：；”这类残渣
    s = re.sub(r"(校区|校园|园区|基地|办学点)\s*[:：]\s*(?=[；;。\.，,]|$)", r"\1", s)

    return _normalize_space(s)

def _should_drop_after_clean(inner_clean: str) -> bool:
    """
    判断括号内容是否应该丢弃：
    - 纯门牌号/路号：丢
    - 清理后为空：丢
    - 不含说明词，且仅为校区/地点/本校等：丢
    """
    t = (inner_clean or "").strip()
    if not t:
        return True

    # 纯门牌号：如 “八一路299号”“西大直街92号”
    if PURE_HOUSE_NO_RE.match(t):
        return True

    # 仍然明显像地址，且没有说明词：丢
    if (not EXPLAIN_WORD_RE.search(t)) and ADDR_STRONG_RE.search(t):
        return True

    # 只剩校区/地点等：丢
    if (not EXPLAIN_WORD_RE.search(t)) and CAMPUS_ONLY_RE.match(re.sub(r"\s+", "", t)):
        return True

    return False

def clean_one_paren_segment(seg: str) -> str:
    """
    对单个括号段进行清洗：
    - 删除地址（省市区路号园区邮编等）
    - 保留说明（只招/录取/建议/报考/毕业/外语/体检等）
    - 校区/地点/本校/本部 这类“无说明”丢弃
    """
    seg = seg.strip()
    if not seg:
        return ""

    left = seg[0]
    right = seg[-1]
    inner = seg[1:-1].strip()

    inner2 = _strip_address_inside_text(inner)

    # 去掉重复院校名括号：如 (吉林大学)、(本校) 这类没说明的一律丢弃
    # （这里不靠“具体院校名”，靠“无说明 + 很短/像机构名”丢弃）
    if _should_drop_after_clean(inner2):
        return ""

    return f"{left}{inner2}{right}"

def clean_school_cell(text) -> str:
    """
    主清洗函数：输入院校字段原字符串，输出“去院校名后的括号说明”（可能为空）
    """
    if text is None:
        return ""

    s = str(text).strip()
    if not s:
        return ""

    # 拆顶层括号，逐段处理括号内容
    parts = parse_segments_keep_parens(s)
    out = []
    for is_paren, seg in parts:
        if not is_paren:
            out.append(seg)
        else:
            out.append(clean_one_paren_segment(seg))

    cleaned = "".join(out)
    cleaned = _normalize_space(cleaned)

    # 全局括号去重
    cleaned = dedupe_parens_globally(cleaned)

    # 去院校名：只保留从第一个括号开始
    cleaned = remove_school_prefix_keep_from_first_paren(cleaned)

    # 最终兜底：去掉“空括号”“残渣括号”
    cleaned = cleaned.replace("()", "").replace("（）", "")
    cleaned = re.sub(r"\(\s*\)", "", cleaned)
    cleaned = re.sub(r"（\s*）", "", cleaned)

    # 如果最后只剩标点/空白，视为空
    if not re.sub(r"[\s，,;；:：。\.()（）+\-—]", "", cleaned):
        return ""

    return cleaned


# =======================
# ④ MajorSubject 拆分：专业名称 + 专业备注
# =======================
def split_major_subject(text):
    """
    专业名称：第一个 '(' 或 '（' 之前
    专业备注：第一个 '(' 或 '（' 及其后所有内容
    若无括号：备注为空
    """
    if text is None:
        return "", ""

    s = str(text).strip()
    idx1 = s.find("(")
    idx2 = s.find("（")
    idxs = [i for i in [idx1, idx2] if i != -1]

    if not idxs:
        return s, ""  # 只有专业名称

    cut = min(idxs)
    name = s[:cut].strip()
    remark = s[cut:].strip()
    return name, remark


# =======================
# ⑤ 合并“院校清洗结果” + “专业备注” 并去重
# =======================
def parse_segments_keep_parens_for_merge(s: str):
    return parse_segments_keep_parens(s)

def merge_remarks_dedupe(remark_a: str, remark_b: str) -> str:
    """
    合并两段备注，并去重：
    - 按顶层括号段去重（括号内内容完全相同，只保留一次）
    - 普通文本也去重（完全一致才去掉）
    - 保留原始顺序：先 a 后 b
    """
    a = (remark_a or "").strip()
    b = (remark_b or "").strip()
    if not a and not b:
        return ""

    combined = (a + b).strip()
    if not combined:
        return ""

    parts = parse_segments_keep_parens_for_merge(combined)

    seen_paren = set()
    seen_text = set()
    out = []

    for is_paren, seg in parts:
        if is_paren:
            inner = seg[1:-1].strip()
            key = re.sub(r"\s+", "", inner)
            if key in seen_paren:
                continue
            seen_paren.add(key)
            out.append(seg)
        else:
            txt = seg
            txt_key = re.sub(r"\s+", " ", txt).strip()
            if txt_key == "":
                out.append(txt)
                continue
            if txt_key in seen_text:
                continue
            seen_text.add(txt_key)
            out.append(txt)

    merged = "".join(out)
    merged = re.sub(r"\s+", " ", merged).strip()
    return merged


# =======================
# ⑥ Excel 工具函数
# =======================
def find_header_col(ws, header_name: str) -> int:
    for col in range(1, ws.max_column + 1):
        val = ws.cell(row=1, column=col).value
        if val is not None and str(val).strip() == header_name:
            return col
    raise ValueError(f"未找到表头列：{header_name}")


# =======================
# ⑦ 主流程：插列并写入
# =======================
def main():
    wb = load_workbook(INPUT_FILE)
    ws = wb[SHEET_NAME] if SHEET_NAME else wb.active

    school_src_col = find_header_col(ws, SCHOOL_SRC_HEADER)
    major_src_col = find_header_col(ws, MAJOR_SRC_HEADER)

    # 在 MajorSubject 后面插入两列：专业名称、专业备注
    major_name_col = major_src_col + 1
    ws.insert_cols(major_name_col, amount=2)
    major_remark_col = major_src_col + 2

    ws.cell(row=1, column=major_name_col).value = MAJOR_NAME_HEADER
    ws.cell(row=1, column=major_remark_col).value = MAJOR_REMARK_HEADER

    # 由于插入列可能影响“院校源列”的索引，如果院校列在 MajorSubject 右侧，需要修正
    if school_src_col >= major_name_col:
        school_src_col += 2

    # 在院校源列后插入一列：院校清洗结果
    school_clean_col = school_src_col + 1
    ws.insert_cols(school_clean_col, amount=1)
    ws.cell(row=1, column=school_clean_col).value = SCHOOL_CLEAN_HEADER

    # 插入院校清洗列后，如果 MajorSubject 在其右侧，需要修正后续引用列
    if major_src_col >= school_clean_col:
        major_src_col += 1
        major_name_col += 1
        major_remark_col += 1

    # 在“专业备注”后插入一列：合并去重后的专业备注
    major_merged_col = major_remark_col + 1
    ws.insert_cols(major_merged_col, amount=1)
    ws.cell(row=1, column=major_merged_col).value = MAJOR_REMARK_MERGED_HEADER

    max_row = ws.max_row

    for r in range(2, max_row + 1):
        school_raw = ws.cell(row=r, column=school_src_col).value
        major_raw = ws.cell(row=r, column=major_src_col).value

        # 1) 院校字段清洗（输出：去院校名后的括号说明）
        school_clean = clean_school_cell(school_raw)
        ws.cell(row=r, column=school_clean_col).value = school_clean

        # 2) MajorSubject 拆分
        major_name, major_remark = split_major_subject(major_raw)
        ws.cell(row=r, column=major_name_col).value = major_name
        ws.cell(row=r, column=major_remark_col).value = major_remark

        # 3) 合并：院校清洗结果 + 专业备注，并去重
        merged_remark = merge_remarks_dedupe(school_clean, major_remark)
        ws.cell(row=r, column=major_merged_col).value = merged_remark

    wb.save(OUTPUT_FILE)
    print("处理完成，输出文件：", OUTPUT_FILE)


if __name__ == "__main__":
    main()

处理完成，输出文件： 地址去除/输出结果/2024-云南.xlsx
